# Append vs Update Benchmark

This notebook benchmarks the time it takes to append vs update 1000 rows in the v2_pred_patch table.

In [1]:
import psycopg2
import time
import random
from datetime import datetime

# Database connection parameters
DB_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"

# Connect to database
conn = psycopg2.connect(DB_URL)
cur = conn.cursor()

## Setup: Create test data

In [2]:
# Generate test data
test_data = []
for i in range(1000):
    patch_uid = random.randint(1000000000000, 9999999999999)
    embed_coords = f'({random.uniform(-100, 100)}, {random.uniform(-100, 100)})'
    grid_cell_i = random.randint(0, 4096)
    grid_cell_j = random.randint(0, 4096)
    pred_label = 99
    patch_coords = f'({random.uniform(-100, 100)}, {random.uniform(-100, 100)})'
    test_data.append((patch_uid, embed_coords, grid_cell_i, grid_cell_j, pred_label, patch_coords))

## Append Benchmark

Measure time to append 1000 rows using INSERT.

In [3]:
# Benchmark append operation (bulk insert)
from psycopg2.extras import execute_values
start_time = time.time()
insert_query = "INSERT INTO v2_pred_patch (patch_uid, embed_coords, grid_cell_i, grid_cell_j, pred_label, patch_coords) VALUES %s"
execute_values(cur, insert_query, test_data, template="(%s, %s::point, %s, %s, %s, %s::point)")
end_time = time.time()
append_time = end_time - start_time
print(f"Append 1000 rows took {append_time:.4f} seconds")
conn.commit()

Append 1000 rows took 0.0609 seconds


## Update Benchmark

Measure time to update 1000 rows using UPDATE.

In [ ]:
# Benchmark update operation (bulk update)
# Generate 1000 random ids between 1 and 700,000,000
random_ids = random.sample(range(1, 700_000_001), 1000)
update_data = []
for id_val in random_ids:
    grid_cell_i = random.randint(0, 1000)
    grid_cell_j = random.randint(0, 1000)
    pred_label = 98
    patch_coords = f'({random.uniform(-100, 100)}, {random.uniform(-100, 100)})'
    update_data.append((grid_cell_i, grid_cell_j, pred_label, patch_coords, id_val))

from psycopg2.extras import execute_values
start_time = time.time()
update_query = """
    UPDATE v2_pred_patch AS t
    SET grid_cell_i = v.ci,
        grid_cell_j = v.cj,
        pred_label = v.pl,
        patch_coords = v.pc::point
    FROM (VALUES %s) AS v(ci, cj, pl, pc, id)
    WHERE t.id = v.id;
"""
execute_values(cur, update_query, update_data, template="(%s, %s, %s, %s, %s)")
end_time = time.time()
update_time = end_time - start_time
print(f"Update 1000 rows took {update_time:.4f} seconds")
conn.commit()

TypeError: not all arguments converted during string formatting

## Results Summary

In [ ]:
print(f"\nResults:\nAppend time: {append_time:.4f} seconds\nUpdate time: {update_time:.4f} seconds")

# Close connection
cur.close()
conn.close()


Results:
Append time: 0.0778 seconds
Update time: 0.2680 seconds
